# Getting Started with Strands Agents for Financial Services

This lab introduces Strands Agents through the lens of financial services. You'll build an AI agent with tools commonly needed by FSI teams — loan calculations, stock lookups, and FX rate checks.

## Overview

In this lab, you will:
- Create your first AI agent with financial tools
- Build custom tools for FSI use cases
- Understand the agent loop and tool calling
- Explore conversation memory within a session

## FSI Context

Imagine you're a TAM supporting a bank or fintech. Your customers ask:
- *"What's the monthly repayment on a $750K mortgage at 6.2%?"*
- *"What's CBA's current stock price?"*
- *"Convert $1M AUD to USD at today's rate"*

An AI agent with the right tools can answer these instantly.

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

⚠️ **Important**: Enable Nova Pro model access in the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess) if you haven't already.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## Building FSI Tools

Let's create three custom tools that a financial services agent would need:

1. **Loan Calculator** — Calculate mortgage/loan repayments
2. **Stock Lookup** — Get current stock prices (ASX/US)
3. **FX Rate** — Currency conversion rates

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator


@tool
def loan_calculator(principal: float, annual_rate: float, years: int) -> str:
    """Calculate monthly loan/mortgage repayment using standard amortization formula.

    Args:
        principal: Loan amount in dollars
        annual_rate: Annual interest rate as percentage (e.g., 6.2 for 6.2%)
        years: Loan term in years
    """
    monthly_rate = (annual_rate / 100) / 12
    num_payments = years * 12
    if monthly_rate == 0:
        monthly_payment = principal / num_payments
    else:
        monthly_payment = principal * (monthly_rate * (1 + monthly_rate)**num_payments) / ((1 + monthly_rate)**num_payments - 1)
    total_paid = monthly_payment * num_payments
    total_interest = total_paid - principal
    return (
        f"Loan: ${principal:,.2f} at {annual_rate}% over {years} years\n"
        f"Monthly repayment: ${monthly_payment:,.2f}\n"
        f"Total interest: ${total_interest:,.2f}\n"
        f"Total paid: ${total_paid:,.2f}"
    )


@tool
def stock_lookup(ticker: str) -> str:
    """Look up current stock price for ASX or US equities.

    Args:
        ticker: Stock ticker symbol (e.g., CBA.AX, BHP.AX, AAPL)
    """
    # Simulated market data for demo purposes
    prices = {
        "CBA.AX": {"price": 112.45, "change": 1.2, "currency": "AUD", "name": "Commonwealth Bank"},
        "BHP.AX": {"price": 45.60, "change": -0.8, "currency": "AUD", "name": "BHP Group"},
        "CSL.AX": {"price": 295.50, "change": 2.1, "currency": "AUD", "name": "CSL Limited"},
        "WBC.AX": {"price": 25.80, "change": 0.5, "currency": "AUD", "name": "Westpac"},
        "NAB.AX": {"price": 32.10, "change": -0.3, "currency": "AUD", "name": "National Australia Bank"},
        "ANZ.AX": {"price": 28.90, "change": 0.7, "currency": "AUD", "name": "ANZ Group"},
        "AAPL": {"price": 198.50, "change": 1.5, "currency": "USD", "name": "Apple Inc"},
        "MSFT": {"price": 425.00, "change": 2.3, "currency": "USD", "name": "Microsoft"},
        "SQ": {"price": 72.30, "change": -1.1, "currency": "USD", "name": "Block Inc (Square)"},
    }
    ticker = ticker.upper()
    if ticker in prices:
        p = prices[ticker]
        direction = "▲" if p["change"] > 0 else "▼"
        return f"{p['name']} ({ticker}): ${p['price']:.2f} {p['currency']} {direction} {abs(p['change'])}%"
    return f"Ticker '{ticker}' not found. Available: {', '.join(prices.keys())}"


@tool
def fx_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    """Get foreign exchange rate and convert currency amount.

    Args:
        from_currency: Source currency code (e.g., AUD, USD, GBP)
        to_currency: Target currency code (e.g., USD, AUD, EUR)
        amount: Amount to convert (default 1.0)
    """
    # Simulated FX rates (base: USD)
    rates_to_usd = {
        "USD": 1.0, "AUD": 0.65, "EUR": 1.08, "GBP": 1.27,
        "JPY": 0.0064, "NZD": 0.60, "SGD": 0.74, "HKD": 0.13,
    }
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    if from_currency not in rates_to_usd or to_currency not in rates_to_usd:
        return f"Currency not found. Available: {', '.join(rates_to_usd.keys())}"
    rate = rates_to_usd[from_currency] / rates_to_usd[to_currency]
    converted = amount * rate
    return f"{amount:,.2f} {from_currency} = {converted:,.2f} {to_currency} (rate: {rate:.4f})"


print("✅ FSI tools defined: loan_calculator, stock_lookup, fx_rate")

## Creating Your FSI Agent

Now let's create an agent with these financial tools. The agent will decide which tool to use based on your question.

In [ ]:
# Create the FSI agent
fsi_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a financial services assistant for a Technical Account Manager (TAM) 
    supporting banking and fintech customers in Australia. You provide concise, accurate 
    financial calculations and market data. Always show your working and cite the tools used.""",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fsi_agent("What's the monthly repayment on a $750,000 mortgage at 6.2% over 30 years?")

In [ ]:
fsi_agent("What's the current price of CBA and how does it compare to Westpac?")

In [ ]:
fsi_agent("Convert $1,000,000 AUD to USD and EUR")

## Understanding the Agent Loop

Let's examine how the agent processes requests — which tools it called, what inputs it provided, and what results it received. This visibility is critical for FSI compliance (audit trail).

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {fsi_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in fsi_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], text[-1] if text else "",
        tool_name[-1] if tool_name else "",
        json.dumps(tool_input[-1], indent=2) if tool_input else "",
        (json.dumps(tool_result[-1], indent=2)[:500]) if tool_result else ""
    )

console.print(table)

## Conversation Memory (Within Session)

The agent remembers context within a session. This is useful for follow-up questions — like a real conversation with a financial advisor.

In [ ]:
fsi_agent("If the rate drops to 5.5%, how much would I save per month on that mortgage?")

In [ ]:
fsi_agent("What have we discussed so far?")

## Memory Limitation: New Session = Blank Slate

If we create a new agent instance, it forgets everything. This is a key limitation we'll solve in Lab 06 with AgentCore Memory.

In [ ]:
# New agent = new session = no memory
fresh_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a financial services assistant. Provide concise responses.",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fresh_agent("What mortgage rate were we discussing?")

## Summary

In this lab, you:

- ✅ Built custom FSI tools (loan calculator, stock lookup, FX rates)
- ✅ Created an agent that autonomously selects the right tool
- ✅ Examined the agent loop (audit trail for compliance)
- ✅ Explored session-based memory and its limitations

### Next Steps

In the following labs, we'll enhance this agent with:
- **Lab 01**: Code Interpreter for dynamic fraud analysis and risk calculations
- **Lab 02**: Browser automation for regulatory monitoring
- **Lab 04**: Deploy tools as managed services (AgentCore Runtime)
- **Lab 05**: Full observability and audit trails
- **Lab 06**: Persistent memory across sessions (remember client preferences)